In [1]:
import csv
import json
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision


runmode = "full"
batchsize = 32
fulltrainingepochs = 30
pilottrainingepochs = 1
fullseedlist = [0, 1]

datapath = "/kaggle/working/data"
outputpath = Path("/kaggle/working/modela_outputs")
checkpointpath = outputpath / "checkpoints"

outputpath.mkdir(parents=True, exist_ok=True)
checkpointpath.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
PyTorch: 2.11.0+cu128
Torchvision: 0.26.0+cu128
GPU: Tesla T4


In [2]:
class convlstmcell(nn.Module):
    def __init__(self, inputchannels, hiddenchannels):
        super().__init__()

        self.gateconvolution = nn.Conv2d(
            inputchannels + hiddenchannels,
            4 * hiddenchannels,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=True,
        )

    def forward(self, inputs, hiddenstate, cellstate):
        gates = self.gateconvolution(
            torch.cat((inputs, hiddenstate), dim=1)
        )

        inputgate, forgetgate, outputgate, candidate = torch.chunk(
            gates, 4, dim=1
        )

        cellstate = (
            torch.sigmoid(forgetgate) * cellstate
            + torch.sigmoid(inputgate) * torch.tanh(candidate)
        )

        hiddenstate = torch.sigmoid(outputgate) * torch.tanh(cellstate)

        return hiddenstate, cellstate


class modela(nn.Module):
    def __init__(self):
        super().__init__()

        self.firstcell = convlstmcell(16, 64)
        self.secondcell = convlstmcell(64, 64)
        self.outputlayer = nn.Conv2d(64, 16, kernel_size=1, bias=True)

    def forward(self, inputframes):
        batchcount = inputframes.shape[0]

        hiddenone = inputframes.new_zeros(batchcount, 64, 16, 16)
        cellone = inputframes.new_zeros(batchcount, 64, 16, 16)

        hiddentwo = inputframes.new_zeros(batchcount, 64, 16, 16)
        celltwo = inputframes.new_zeros(batchcount, 64, 16, 16)

        predictions = []
        previousprediction = None

        for step in range(9):
            if step < 5:
                currentframe = inputframes[:, step]
            else:
                currentframe = previousprediction

            foldedframe = F.pixel_unshuffle(currentframe, 4)

            hiddenone, cellone = self.firstcell(
                foldedframe, hiddenone, cellone
            )

            hiddentwo, celltwo = self.secondcell(
                hiddenone, hiddentwo, celltwo
            )

            previousprediction = torch.sigmoid(
                F.pixel_shuffle(self.outputlayer(hiddentwo), 4)
            )

            if step >= 4:
                predictions.append(previousprediction)

        return torch.stack(predictions, dim=1)


def countparameters(model):
    total = 0

    for parameter in model.parameters():
        if parameter.requires_grad:
            total += parameter.numel()

    return total


checkmodel = modela()
parametercount = countparameters(checkmodel)

print("Model A trainable parameters:", parametercount)

if parametercount != 480784:
    raise RuntimeError(
        f"Parameter count mismatch: {parametercount} != 480784"
    )

print("Architecture check passed.")
del checkmodel

Model A trainable parameters: 480784
Architecture check passed.


In [3]:
def loadvideos():
    dataset = torchvision.datasets.MovingMNIST(
        root=datapath,
        split=None,
        download=True,
    )

    if hasattr(dataset, "data"):
        rawvideos = torch.as_tensor(dataset.data)
    else:
        videos = []

        for index in range(len(dataset)):
            video = dataset[index]

            if isinstance(video, (tuple, list)):
                video = video[0]

            videos.append(torch.as_tensor(video))

        rawvideos = torch.stack(videos)

    if rawvideos.ndim == 4:
        rawvideos = rawvideos.unsqueeze(2)

    expectedshape = (10000, 20, 1, 64, 64)

    if tuple(rawvideos.shape) != expectedshape:
        raise RuntimeError(
            f"Unexpected Moving MNIST shape: {tuple(rawvideos.shape)}"
        )

    rawvideos = rawvideos.contiguous()

    scaledata = (
        not rawvideos.is_floating_point()
        or rawvideos.max().item() > 1.0
    )

    return rawvideos, scaledata


def convertvideos(videos, scaledata):
    videos = videos.to(device=device, dtype=torch.float32)

    if scaledata:
        videos = videos / 255.0

    return videos


print("Loading Moving MNIST...")
rawvideos, scaledata = loadvideos()

trainvideos = rawvideos[:8000]
validationvideos = rawvideos[8000:9000]
testvideos = rawvideos[9000:10000]

print("Training videos:", len(trainvideos))
print("Validation videos:", len(validationvideos))
print("Test videos:", len(testvideos))
print("Scaling:", "divide by 255" if scaledata else "already in [0, 1]")

Loading Moving MNIST...


100%|██████████| 819M/819M [02:23<00:00, 5.72MB/s]  


Training videos: 8000
Validation videos: 1000
Test videos: 1000
Scaling: divide by 255


In [6]:
def mirrorframes(frames):
    return torch.flip(frames, dims=(-1,))


def rotateframes(frames):
    return torch.rot90(frames, k=1, dims=(-2, -1))


def getframeindices(speed):
    inputindices = [step * speed for step in range(5)]

    targetindices = [
        (5 + step) * speed
        for step in range(5)
        if (5 + step) * speed < 20
    ]

    return inputindices, targetindices


testtensor = torch.arange(
    2 * 20 * 64 * 64,
    dtype=torch.float32,
).reshape(2, 20, 64, 64)

doublemirror = mirrorframes(mirrorframes(testtensor))
fourrotations = testtensor.clone()

for step in range(4):
    fourrotations = rotateframes(fourrotations)

mirrorrotationmirror = mirrorframes(
    rotateframes(mirrorframes(testtensor))
)

inverserotation = torch.rot90(
    testtensor,
    k=-1,
    dims=(-2, -1),
)

checks = {
    "doublemirror": torch.equal(doublemirror, testtensor),
    "fourrotations": torch.equal(fourrotations, testtensor),
    "mirrorrotationmirror": torch.equal(
        mirrorrotationmirror,
        inverserotation,
    ),
}

markervideo = torch.zeros(20, 1, 64, 64)
markervideo[7, 0, 10, 20] = 1

mirroredmarker = mirrorframes(markervideo)
rotatedmarker = rotateframes(markervideo)

checks["mirrormarker"] = (
    mirroredmarker[7, 0, 10, 43].item() == 1
    and mirroredmarker.sum().item() == 1
)

checks["rotatedmarker"] = (
    rotatedmarker[7, 0, 43, 10].item() == 1
    and rotatedmarker.sum().item() == 1
)

expectedindices = {
    1: ([0, 1, 2, 3, 4], [5, 6, 7, 8, 9]),
    2: ([0, 2, 4, 6, 8], [10, 12, 14, 16, 18]),
    3: ([0, 3, 6, 9, 12], [15, 18]),
}

for speed in [1, 2, 3]:
    checks[f"speed{speed}"] = (
        getframeindices(speed) == expectedindices[speed]
    )

for name, passed in checks.items():
    print(name, "pass" if passed else "fail")

if not all(checks.values()):
    raise RuntimeError("Table 1 not correct")

with open(outputpath / "table1checks.json", "w") as file:
    json.dump(checks, file, indent=2)

print("Table 1 valid")

doublemirror pass
fourrotations pass
mirrorrotationmirror pass
mirrormarker pass
rotatedmarker pass
speed1 pass
speed2 pass
speed3 pass
Table 1 valid


In [8]:
import cv2


def calculateopticalflow(trainvideos, scaledata):
    directionvalues = []
    speedvalues = []

    for videoindex in range(50):
        video = trainvideos[videoindex]

        if video.ndim == 4:
            video = video[:, 0]

        video = video.cpu().numpy()

        if not scaledata:
            video = video * 255.0

        video = np.clip(video, 0, 255).astype(np.uint8)

        for frameindex in range(19):
            previousframe = video[frameindex]
            nextframe = video[frameindex + 1]

            flow = cv2.calcOpticalFlowFarneback(
                previousframe,
                nextframe,
                None,
                0.5,
                3,
                15,
                3,
                5,
                1.2,
                0,
            )

            horizontalflow = flow[..., 0]
            verticalflow = flow[..., 1]

            magnitude = np.sqrt(
                horizontalflow ** 2 + verticalflow ** 2
            )

            validpixels = (
                (previousframe > 51)
                & (nextframe > 51)
                & (magnitude > 0.5)
            )

            directions = (
                np.degrees(
                    np.arctan2(
                        -verticalflow[validpixels],
                        horizontalflow[validpixels],
                    )
                ) + 360
            ) % 360

            directionvalues.append(directions)
            speedvalues.append(magnitude[validpixels])

    directionvalues = np.concatenate(directionvalues)
    speedvalues = np.concatenate(speedvalues)

    if len(speedvalues) == 0:
        raise RuntimeError("No valid optical-flow pixels found.")

    medianspeed = float(np.median(speedvalues))

    figure, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    directioncounts, directionedges = np.histogram(
        directionvalues,
        bins=18,
        range=(0, 360),
    )

    directionfractions = directioncounts / directioncounts.sum()

    axes[0].bar(
        directionedges[:-1],
        directionfractions,
        width=20,
        align="edge",
    )

    axes[0].axhline(
        1 / 18,
        linestyle="--",
        label="Uniform: 1/18",
    )

    axes[0].set_xlabel("Direction (degrees)")
    axes[0].set_ylabel("Fraction of kept pixels")
    axes[0].set_title("Optical-flow direction")
    axes[0].legend()

    axes[1].hist(
        speedvalues,
        bins=50,
        weights=np.ones_like(speedvalues) / len(speedvalues),
    )

    axes[1].axvline(
        medianspeed,
        linestyle="--",
        label=f"Median = {medianspeed:.3f} px/frame",
    )

    axes[1].set_xlabel("Speed (pixels/frame)")
    axes[1].set_ylabel("Fraction of kept pixels")
    axes[1].set_title("Optical-flow speed")
    axes[1].legend()

    figure.suptitle(
        f"First 50 training videos | {len(speedvalues):,} kept pixels"
    )

    figure.tight_layout()
    figure.savefig(
        outputpath / "figure1_opticalflow.png",
        dpi=180,
        bbox_inches="tight",
    )
    plt.close(figure)

    statistics = {
        "videos": 50,
        "framepairs": 950,
        "keptpixels": int(len(speedvalues)),
        "medianspeed": medianspeed,
        "directionfractions": directionfractions.tolist(),
        "directionedges": directionedges.tolist(),
    }

    with open(
        outputpath / "figure1_statistics.json",
        "w",
    ) as file:
        json.dump(statistics, file, indent=2)

    print("Kept pixels:", statistics["keptpixels"])
    print("Median speed:", medianspeed)
    print("Figure 1 saved.")

    return statistics


flowstatistics = calculateopticalflow(trainvideos, scaledata)

Kept pixels: 103050
Median speed: 3.317502498626709
Figure 1 saved.


In [9]:
def setseed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def trainmodel(seed, epochcount, trainvideos, scaledata, pilot):
    setseed(seed)

    model = modela().to(device)
    parametercount = countparameters(model)

    if parametercount != 480784:
        raise RuntimeError(
            f"Model A parameter count mismatch: {parametercount}"
        )

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    epochseconds = []
    epochlosses = []

    print(
        f"\nModel A | seed {seed} | "
        f"parameters {parametercount} | epochs {epochcount}"
    )

    for epoch in range(epochcount):
        model.train()

        videoorder = torch.randperm(trainvideos.shape[0])
        epochstart = time.perf_counter()
        losssum = 0.0
        batchcount = 0

        for batchstart in range(0, trainvideos.shape[0], batchsize):
            batchindices = videoorder[batchstart:batchstart + batchsize]

            batchvideos = convertvideos(
                trainvideos[batchindices],
                scaledata,
            )

            currentbatch = batchvideos.shape[0]

            startframes = torch.randint(
                0, 11, (currentbatch,), device=device
            )

            batchpositions = torch.arange(currentbatch, device=device)
            inputoffsets = torch.arange(5, device=device)
            targetoffsets = torch.arange(5, 10, device=device)

            inputframes = batchvideos[
                batchpositions[:, None],
                startframes[:, None] + inputoffsets[None, :],
            ]

            targetframes = batchvideos[
                batchpositions[:, None],
                startframes[:, None] + targetoffsets[None, :],
            ]

            predictions = model(inputframes)
            loss = F.mse_loss(predictions, targetframes)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

            losssum += loss.detach().item()
            batchcount += 1

        if device.type == "cuda":
            torch.cuda.synchronize()

        elapsed = time.perf_counter() - epochstart
        epochloss = losssum / batchcount

        epochseconds.append(elapsed)
        epochlosses.append(epochloss)

        print(
            f"Epoch {epoch + 1:02d}/{epochcount} | "
            f"loss {epochloss:.6f} | {elapsed:.2f}s"
        )

    checkpointfile = checkpointpath / (
        f"modelapilotseed{seed}.pt"
        if pilot
        else f"modelaseed{seed}.pt"
    )

    savedstate = {}

    for name, value in model.state_dict().items():
        savedstate[name] = value.detach().cpu()

    trainingrecord = {
        "seed": seed,
        "epochs": epochcount,
        "parameters": parametercount,
        "meanepochseconds": float(np.mean(epochseconds)),
        "epochseconds": epochseconds,
        "epochlosses": epochlosses,
        "checkpoint": str(checkpointfile),
    }

    torch.save(
        {
            "modelstate": savedstate,
            **trainingrecord,
        },
        checkpointfile,
    )

    del optimizer, model

    if device.type == "cuda":
        torch.cuda.empty_cache()

    return trainingrecord

In [11]:
def evaluatemodel(testvideos, scaledata, model=None, method="learned"):
    if model is not None:
        model.eval()

    squaredtotals = torch.zeros(5, dtype=torch.float64)
    inferenceseconds = 0.0
    testedvideos = 0

    with torch.no_grad():
        for batchstart in range(0, testvideos.shape[0], batchsize):
            batchvideos = convertvideos(
                testvideos[batchstart:batchstart + batchsize],
                scaledata,
            )

            inputframes = batchvideos[:, :5]
            targetframes = batchvideos[:, 5:10]

            if method == "blank":
                predictions = torch.zeros_like(targetframes)

            elif method == "copylast":
                predictions = inputframes[:, -1:].expand_as(targetframes)

            else:
                if device.type == "cuda":
                    torch.cuda.synchronize()

                inferencestart = time.perf_counter()
                predictions = model(inputframes)

                if device.type == "cuda":
                    torch.cuda.synchronize()

                inferenceseconds += time.perf_counter() - inferencestart

            squaredtotals += (
                (predictions - targetframes).square()
                .sum(dim=(0, 2, 3, 4))
                .detach()
                .cpu()
                .double()
            )

            testedvideos += targetframes.shape[0]

    horizonerrors = (
        squaredtotals / (testedvideos * 64 * 64)
    ).tolist()

    return {
        "mse1": horizonerrors[0],
        "mse2": horizonerrors[1],
        "mse3": horizonerrors[2],
        "mse4": horizonerrors[3],
        "mse5": horizonerrors[4],
        "e5": float(np.mean(horizonerrors)),
        "parameters": countparameters(model) if model is not None else 0,
        "inferenceseconds": (
            inferenceseconds if model is not None else None
        ),
        "videos": testedvideos,
    }


def makefigure2(model, testvideos, scaledata, selectedindices):
    if len(selectedindices) != 3:
        raise RuntimeError("Not three test-video indices.")

    for localindex in selectedindices:
        if localindex < 0 or localindex >= len(testvideos):
            raise RuntimeError(f"Invalid test-video index: {localindex}")

    figure, axes = plt.subplots(3, 15, figsize=(18, 4.5))
    model.eval()

    with torch.no_grad():
        for row, localindex in enumerate(selectedindices):
            video = convertvideos(
                testvideos[localindex:localindex + 1],
                scaledata,
            )

            inputframes = video[:, :5]
            targetframes = video[:, 5:10]
            predictions = model(inputframes)

            framegroups = [
                inputframes[0, :, 0].cpu(),
                predictions[0, :, 0].cpu(),
                targetframes[0, :, 0].cpu(),
            ]

            for groupindex, frames in enumerate(framegroups):
                for frameindex in range(5):
                    column = groupindex * 5 + frameindex

                    axes[row, column].imshow(
                        frames[frameindex].numpy(),
                        cmap="gray",
                        vmin=0,
                        vmax=1,
                    )

                    axes[row, column].axis("off")

                    if row == 0 and frameindex == 2:
                        axes[row, column].set_title(
                            ["Input", "Prediction", "Ground truth"][
                                groupindex
                            ],
                            fontsize=10,
                        )

            axes[row, 0].set_ylabel(
                f"Test {9000 + localindex}",
                fontsize=8,
            )

    figure.tight_layout()

    figurefile = outputpath / "figure2modela.png"
    figure.savefig(figurefile, dpi=180, bbox_inches="tight")
    plt.close(figure)

    print("Figure saved:", figurefile)

In [12]:
if runmode == "smoke":
    seedlist = [0]
    epochcount = pilottrainingepochs

elif runmode == "full":
    seedlist = fullseedlist
    epochcount = fulltrainingepochs

else:
    raise RuntimeError("Set runmode to either 'pilot' or 'full'.")


trainingrecords = []

for currentseed in seedlist:
    trainingrecord = trainmodel(
        currentseed,
        epochcount,
        trainvideos,
        scaledata,
        runmode == "pilot",
    )

    trainingrecords.append(trainingrecord)

    with open(
        outputpath / "traininglog.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(trainingrecords, file, indent=2)

print("\nTraining run complete.")
print("Saved outputs:", outputpath)


Model A | seed 0 | parameters 480784 | epochs 30
Epoch 01/30 | loss 0.046084 | 11.81s
Epoch 02/30 | loss 0.039851 | 10.22s
Epoch 03/30 | loss 0.038787 | 10.31s
Epoch 04/30 | loss 0.037166 | 10.41s
Epoch 05/30 | loss 0.035215 | 10.52s
Epoch 06/30 | loss 0.033449 | 10.61s
Epoch 07/30 | loss 0.031971 | 10.70s
Epoch 08/30 | loss 0.030789 | 10.81s
Epoch 09/30 | loss 0.030127 | 10.93s
Epoch 10/30 | loss 0.029612 | 11.07s
Epoch 11/30 | loss 0.029057 | 11.19s
Epoch 12/30 | loss 0.028610 | 11.25s
Epoch 13/30 | loss 0.028146 | 11.11s
Epoch 14/30 | loss 0.027771 | 11.03s
Epoch 15/30 | loss 0.027238 | 11.01s
Epoch 16/30 | loss 0.026744 | 11.01s
Epoch 17/30 | loss 0.026224 | 11.04s
Epoch 18/30 | loss 0.025644 | 11.08s
Epoch 19/30 | loss 0.025045 | 11.11s
Epoch 20/30 | loss 0.024582 | 11.12s
Epoch 21/30 | loss 0.023976 | 11.11s
Epoch 22/30 | loss 0.023543 | 11.10s
Epoch 23/30 | loss 0.023154 | 11.07s
Epoch 24/30 | loss 0.022716 | 11.08s
Epoch 25/30 | loss 0.022311 | 11.07s
Epoch 26/30 | loss 0.0219

In [13]:
if runmode != "full":
    raise RuntimeError("Training not completed")

testresults = []

for currentseed in fullseedlist:
    checkpointfile = checkpointpath / f"modelaseed{currentseed}.pt"
    checkpoint = torch.load(checkpointfile, map_location="cpu")

    model = modela().to(device)
    model.load_state_dict(checkpoint["modelstate"])

    result = evaluatemodel(
        testvideos,
        scaledata,
        model,
        "learned",
    )

    result["model"] = "Model A"
    result["seed"] = currentseed
    result["meanepochseconds"] = checkpoint["meanepochseconds"]

    testresults.append(result)

    print(
        f"Model A seed {currentseed} | "
        f"E5 {result['e5']:.6f} | "
        f"inference {result['inferenceseconds']:.3f}s"
    )

    if currentseed == 0:
        figure2indices = [0, 1, 2]
        makefigure2(model, testvideos, scaledata, figure2indices)

    del model, checkpoint

    if device.type == "cuda":
        torch.cuda.empty_cache()


for method, label in [
    ("blank", "Blank"),
    ("copylast", "Copy-last"),
]:
    result = evaluatemodel(
        testvideos,
        scaledata,
        None,
        method,
    )

    result["model"] = label
    result["seed"] = "-"
    result["meanepochseconds"] = None

    testresults.append(result)

    print(f"{label} | E5 {result['e5']:.6f}")


seedzero = next(
    row for row in testresults
    if row["model"] == "Model A" and row["seed"] == 0
)

seedone = next(
    row for row in testresults
    if row["model"] == "Model A" and row["seed"] == 1
)

meanrow = {
    "model": "Model A mean",
    "seed": "mean(0,1)",
    "parameters": 480784,
    "videos": 1000,
    "inferenceseconds": float(np.mean([
        seedzero["inferenceseconds"],
        seedone["inferenceseconds"],
    ])),
    "meanepochseconds": float(np.mean([
        seedzero["meanepochseconds"],
        seedone["meanepochseconds"],
    ])),
}

for metric in ["mse1", "mse2", "mse3", "mse4", "mse5", "e5"]:
    meanrow[metric] = float(np.mean([
        seedzero[metric],
        seedone[metric],
    ]))

testresults.insert(2, meanrow)

with open(
    outputpath / "table2results.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(testresults, file, indent=2)

fieldnames = [
    "model",
    "seed",
    "mse1",
    "mse2",
    "mse3",
    "mse4",
    "mse5",
    "e5",
    "parameters",
    "meanepochseconds",
    "inferenceseconds",
    "videos",
]

with open(
    outputpath / "table2results.csv",
    "w",
    newline="",
    encoding="utf-8",
) as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(testresults)

print("\nTable 2 results")

for row in testresults:
    print(
        row["model"],
        "| seed:", row["seed"],
        "| MSE:", [
            f"{row[f'mse{horizon}']:.6f}"
            for horizon in range(1, 6)
        ],
        "| E5:", f"{row['e5']:.6f}",
        "| parameters:", row["parameters"],
    )

print("\nResults saved in:", outputpath)
print("Checkpoints saved in:", checkpointpath)

Model A seed 0 | E5 0.021112 | inference 0.516s
Figure saved: /kaggle/working/modela_outputs/figure2modela.png
Model A seed 1 | E5 0.021100 | inference 0.402s
Blank | E5 0.042564
Copy-last | E5 0.064082

Table 2 results
Model A | seed: 0 | MSE: ['0.015943', '0.019300', '0.021636', '0.023441', '0.025242'] | E5: 0.021112 | parameters: 480784
Model A | seed: 1 | MSE: ['0.016030', '0.019353', '0.021645', '0.023372', '0.025101'] | E5: 0.021100 | parameters: 480784
Model A mean | seed: mean(0,1) | MSE: ['0.015987', '0.019327', '0.021641', '0.023406', '0.025171'] | E5: 0.021106 | parameters: 480784
Blank | seed: - | MSE: ['0.042610', '0.042570', '0.042576', '0.042532', '0.042530'] | E5: 0.042564 | parameters: 0
Copy-last | seed: - | MSE: ['0.049164', '0.060420', '0.066889', '0.070779', '0.073160'] | E5: 0.064082 | parameters: 0

Results saved in: /kaggle/working/modela_outputs
Checkpoints saved in: /kaggle/working/modela_outputs/checkpoints


In [14]:
def transformframes(frames, transformname):
    if transformname == "mirror":
        return torch.flip(frames, dims=(-1,))

    if transformname == "rotation":
        return torch.rot90(frames, k=1, dims=(-2, -1))

    raise ValueError(f"Unknown transformation: {transformname}")


def predictframes(inputs, model=None, method="learned"):
    if method == "blank":
        return torch.zeros_like(inputs)

    if method == "copylast":
        return inputs[:, -1:].expand(-1, 5, -1, -1, -1)

    if model is None:
        raise RuntimeError("Learned prediction requires a model.")

    return model(inputs)


def loadmodelacheckpoint(seed):
    checkpointfile = checkpointpath / f"modelaseed{seed}.pt"

    if not checkpointfile.exists():
        raise FileNotFoundError(
            f"Missing Model A checkpoint: {checkpointfile}"
        )

    checkpoint = torch.load(
        checkpointfile,
        map_location="cpu",
        weights_only=True,
    )

    loadedmodel = modela().to(device)
    loadedmodel.load_state_dict(checkpoint["modelstate"])
    loadedmodel.eval()

    return loadedmodel


def evaluatetransformations(model=None, method="learned"):
    if model is not None:
        model.eval()

    collected = {
        "e5": [],
        "cm": [],
        "em": [],
        "cr": [],
        "er": [],
    }

    with torch.inference_mode():
        for batchstart in range(0, len(testvideos), batchsize):
            videos = convertvideos(
                testvideos[batchstart:batchstart + batchsize],
                scaledata,
            )

            inputs = videos[:, :5]
            targets = videos[:, 5:10]

            original = predictframes(inputs, model, method)

            collected["e5"].append(
                (original - targets)
                .square()
                .mean(dim=(1, 2, 3, 4))
                .cpu()
            )

            for name, consistencykey, errorkey in [
                ("mirror", "cm", "em"),
                ("rotation", "cr", "er"),
            ]:
                transformedinputs = transformframes(inputs, name)
                transformedtargets = transformframes(targets, name)

                transformedpredictions = predictframes(
                    transformedinputs, model, method
                )

                transformedoriginal = transformframes(original, name)

                consistency = (
                    transformedpredictions - transformedoriginal
                ).square().mean(dim=(1, 2, 3, 4))

                transformederror = (
                    transformedpredictions - transformedtargets
                ).square().mean(dim=(1, 2, 3, 4))

                collected[consistencykey].append(
                    consistency.cpu()
                )

                collected[errorkey].append(
                    transformederror.cpu()
                )

    pervideo = {
        key: torch.cat(values).numpy()
        for key, values in collected.items()
    }

    summary = {
        key: float(values.mean())
        for key, values in pervideo.items()
    }

    return summary, pervideo


transformationrecords = []
pervideorecords = {}

for seed in fullseedlist:
    currentmodel = loadmodelacheckpoint(seed)

    summary, pervideo = evaluatetransformations(currentmodel)

    transformationrecords.append({
        "model": "Model A",
        "seed": seed,
        **summary,
    })

    pervideorecords[seed] = pervideo

    print(f"Model A seed {seed}:", summary)

    del currentmodel
    if device.type == "cuda":
        torch.cuda.empty_cache()


for method, label in [
    ("blank", "Blank"),
    ("copylast", "Copy-last"),
]:
    summary, _ = evaluatetransformations(
        model=None,
        method=method,
    )

    transformationrecords.append({
        "model": label,
        "seed": "-",
        **summary,
    })

    print(label, summary)


table3 = []

for label in ["Model A", "Blank", "Copy-last"]:
    rows = [
        row for row in transformationrecords
        if row["model"] == label
    ]

    result = {"model": label}

    for metric in ["e5", "cm", "em", "cr", "er"]:
        result[metric] = float(np.mean([
            row[metric] for row in rows
        ]))

    table3.append(result)


with open(outputpath / "table3results.json", "w") as file:
    json.dump(table3, file, indent=2)

with open(
    outputpath / "table3results.csv",
    "w",
    newline="",
) as file:
    writer = csv.DictWriter(
        file,
        fieldnames=list(table3[0].keys()),
    )
    writer.writeheader()
    writer.writerows(table3)


with open(
    outputpath / "table3perseed.json",
    "w",
) as file:
    json.dump(transformationrecords, file, indent=2)


print("\nTABLE 3")
for row in table3:
    print(row)

Model A seed 0: {'e5': 0.02111237309873104, 'cm': 0.002498210407793522, 'em': 0.021969031542539597, 'cr': 0.004451564978808165, 'er': 0.023322686553001404}
Model A seed 1: {'e5': 0.021100342273712158, 'cm': 0.0022981567308306694, 'em': 0.021832183003425598, 'cr': 0.003319942858070135, 'er': 0.02293422818183899}
Blank {'e5': 0.04256360977888107, 'cm': 0.0, 'em': 0.04256360977888107, 'cr': 0.0, 'er': 0.04256360977888107}
Copy-last {'e5': 0.06408222764730453, 'cm': 0.0, 'em': 0.06408222764730453, 'cr': 0.0, 'er': 0.06408222764730453}

TABLE 3
{'model': 'Model A', 'e5': 0.0211063576862216, 'cm': 0.0023981835693120956, 'em': 0.021900607272982597, 'cr': 0.00388575391843915, 'er': 0.023128457367420197}
{'model': 'Blank', 'e5': 0.04256360977888107, 'cm': 0.0, 'em': 0.04256360977888107, 'cr': 0.0, 'er': 0.04256360977888107}
{'model': 'Copy-last', 'e5': 0.06408222764730453, 'cm': 0.0, 'em': 0.06408222764730453, 'cr': 0.0, 'er': 0.06408222764730453}


In [15]:
def evaluatespeed(speed, model=None, method="learned"):
    inputindices, targetindices = getframeindices(speed)

    if len(targetindices) < 2:
        raise RuntimeError(
            f"Speed {speed} has fewer than two valid targets."
        )

    if model is not None:
        model.eval()

    allerrors = []

    with torch.inference_mode():
        for batchstart in range(0, len(testvideos), batchsize):
            videos = convertvideos(
                testvideos[batchstart:batchstart + batchsize],
                scaledata,
            )

            inputs = videos[:, inputindices]
            targets = videos[:, targetindices]

            predictions = predictframes(
                inputs, model, method
            )[:, :len(targetindices)]

            errors = (
                (predictions - targets)
                .square()
                .mean(dim=(2, 3, 4))
            )

            allerrors.append(errors.cpu())

    allerrors = torch.cat(allerrors)

    return {
        "e2": float(allerrors[:, :2].mean()),
        "e5": (
            float(allerrors[:, :5].mean())
            if len(targetindices) >= 5
            else None
        ),
        "horizonerrors": [
            float(allerrors[:, index].mean())
            for index in range(allerrors.shape[1])
        ],
    }


speedrecords = []

for seed in fullseedlist:
    currentmodel = loadmodelacheckpoint(seed)

    for speed in [1, 2, 3]:
        result = evaluatespeed(
            speed,
            model=currentmodel,
        )

        speedrecords.append({
            "model": "Model A",
            "seed": seed,
            "speed": speed,
            **result,
        })

        print(
            f"Model A seed {seed} | {speed}x | "
            f"E2 {result['e2']:.6f} | "
            f"E5 {result['e5']}"
        )

    del currentmodel
    if device.type == "cuda":
        torch.cuda.empty_cache()


for speed in [1, 2, 3]:
    result = evaluatespeed(
        speed,
        method="copylast",
    )

    speedrecords.append({
        "model": "Copy-last",
        "seed": "-",
        "speed": speed,
        **result,
    })


table4 = []

for label in ["Model A", "Copy-last"]:
    row = {"model": label}

    for speed in [1, 2, 3]:
        matching = [
            result for result in speedrecords
            if result["model"] == label
            and result["speed"] == speed
        ]

        row[f"e2_{speed}x"] = float(np.mean([
            result["e2"] for result in matching
        ]))

        if speed in [1, 2]:
            row[f"e5_{speed}x"] = float(np.mean([
                result["e5"] for result in matching
            ]))

    table4.append(row)


with open(outputpath / "table4results.json", "w") as file:
    json.dump(table4, file, indent=2)

with open(
    outputpath / "table4results.csv",
    "w",
    newline="",
) as file:
    writer = csv.DictWriter(
        file,
        fieldnames=list(table4[0].keys()),
    )
    writer.writeheader()
    writer.writerows(table4)


with open(outputpath / "table4perseed.json", "w") as file:
    json.dump(speedrecords, file, indent=2)


print("\nTABLE 4")
for row in table4:
    print(row)

Model A seed 0 | 1x | E2 0.017621 | E5 0.02111237123608589
Model A seed 0 | 2x | E2 0.043992 | E5 0.0487440750002861
Model A seed 0 | 3x | E2 0.046108 | E5 None
Model A seed 1 | 1x | E2 0.017692 | E5 0.021100344136357307
Model A seed 1 | 2x | E2 0.044608 | E5 0.04871988669037819
Model A seed 1 | 3x | E2 0.050453 | E5 None

TABLE 4
{'model': 'Model A', 'e2_1x': 0.017656604759395123, 'e5_1x': 0.0211063576862216, 'e2_2x': 0.04430012218654156, 'e5_2x': 0.048731980845332146, 'e2_3x': 0.04828069172799587}
{'model': 'Copy-last', 'e2_1x': 0.054791729897260666, 'e5_1x': 0.06408222764730453, 'e2_2x': 0.06606846302747726, 'e5_2x': 0.07188228517770767, 'e2_3x': 0.07025302946567535}


In [17]:
from io import BytesIO
from PIL import Image


visualmodel = loadmodelacheckpoint(0)

selectedexamples = {
    "mirror": np.argsort(
        pervideorecords[0]["cm"]
    )[-2:][::-1].tolist(),

    "rotation": np.argsort(
        pervideorecords[0]["cr"]
    )[-2:][::-1].tolist(),
}


def makeconsistencyfigure(videoindex, transformname, horizon=None):
    videos = convertvideos(
        testvideos[videoindex:videoindex + 1],
        scaledata,
    )

    inputs = videos[:, :5]
    targets = videos[:, 5:10]

    transformedinputs = transformframes(inputs, transformname)
    transformedtargets = transformframes(targets, transformname)

    with torch.inference_mode():
        original = visualmodel(inputs)
        transformed = visualmodel(transformedinputs)

    transformedoriginal = transformframes(
        original, transformname
    )

    rows = [
        ("Original", inputs, original),
        ("Transformed truth", transformedinputs, transformedtargets),
        ("f(T(X))", transformedinputs, transformed),
        ("T(f(X))", transformedinputs, transformedoriginal),
    ]

    figure, axes = plt.subplots(
        4, 10,
        figsize=(15, 6),
        squeeze=False,
    )

    for rowindex, (label, past, future) in enumerate(rows):
        for column in range(10):
            axis = axes[rowindex, column]

            if column < 5:
                frame = past[0, column, 0]
            elif horizon is None or column == horizon + 5:
                frame = future[0, column - 5, 0]
            else:
                axis.axis("off")
                continue

            axis.imshow(
                frame.detach().cpu().numpy(),
                cmap="gray",
                vmin=0,
                vmax=1,
            )
            axis.set_xticks([])
            axis.set_yticks([])

            if rowindex == 0:
                axis.set_title(
                    f"Input {column + 1}"
                    if column < 5
                    else f"Future {column - 4}",
                    fontsize=8,
                )

        axes[rowindex, 0].set_ylabel(label, fontsize=8)

    figure.suptitle(
        f"{transformname.title()} | Test video {9000 + videoindex}"
    )
    figure.tight_layout()

    return figure


for transformname, indices in selectedexamples.items():
    for rank, videoindex in enumerate(indices, start=1):
        filename = (
            f"figure3_{transformname}_rank{rank}_"
            f"test{9000 + videoindex}"
        )

        figure = makeconsistencyfigure(
            videoindex, transformname
        )

        figure.savefig(
            outputpath / f"{filename}.png",
            dpi=150,
            bbox_inches="tight",
        )
        plt.close(figure)

        gifframes = []

        for horizon in range(5):
            figure = makeconsistencyfigure(
                videoindex,
                transformname,
                horizon,
            )

            buffer = BytesIO()
            figure.savefig(
                buffer,
                format="png",
                dpi=75,
                bbox_inches="tight",
            )
            plt.close(figure)

            buffer.seek(0)
            gifframes.append(
                Image.open(buffer).convert("RGB").copy()
            )
            buffer.close()

        gifframes[0].save(
            outputpath / f"{filename}.gif",
            save_all=True,
            append_images=gifframes[1:],
            duration=650,
            loop=0,
        )

        for frame in gifframes:
            frame.close()

        print("Saved:", filename)


figure, axes = plt.subplots(
    2, 2,
    figsize=(12, 8),
    squeeze=False,
)

for rowindex, seed in enumerate(fullseedlist):
    for column, (metric, name) in enumerate([
        ("cm", "Mirror"),
        ("cr", "Rotation"),
    ]):
        axis = axes[rowindex, column]
        values = pervideorecords[seed][metric]

        axis.hist(values, bins=40)
        axis.set_title(f"{name} consistency | Seed {seed}")
        axis.set_xlabel("Per-video consistency MSE")
        axis.set_ylabel("Number of videos")

        if seed == 0:
            for videoindex in selectedexamples[
                "mirror" if metric == "cm" else "rotation"
            ]:
                axis.axvline(
                    values[videoindex],
                    linestyle="--",
                    label=f"Test {9000 + videoindex}",
                )
            axis.legend(fontsize=8)

figure.tight_layout()
figure.savefig(
    outputpath / "figure4_consistency_histograms.png",
    dpi=180,
    bbox_inches="tight",
)
plt.close(figure)


with open(
    outputpath / "figure3_selectedexamples.json",
    "w",
) as file:
    json.dump(
        {
            name: [
                9000 + index for index in indices
            ]
            for name, indices in selectedexamples.items()
        },
        file,
        indent=2,
    )

print("Figures and GIFs saved.")

Saved: figure3_mirror_rank1_test9269
Saved: figure3_mirror_rank2_test9347
Saved: figure3_rotation_rank1_test9733
Saved: figure3_rotation_rank2_test9883
Figures and GIFs saved.


In [19]:
import shutil
from pathlib import Path
from zipfile import ZipFile


outputpath = Path("/kaggle/working/modela_outputs")
archivepath = Path("/kaggle/working/modela_results_bundle.zip")

if not outputpath.exists():
    raise FileNotFoundError(
        f"Model A output directory does not exist: {outputpath}"
    )


architecturefile = outputpath / "modela_architecture.py"

if architecturefile.exists():
    architecturefile.unlink()


for seed in [0, 1]:
    checkpointfile = outputpath / "checkpoints" / f"modelaseed{seed}.pt"

    if not checkpointfile.exists():
        raise FileNotFoundError(
            f"Missing Model A checkpoint for seed {seed}: {checkpointfile}"
        )


shutil.make_archive(
    str(archivepath.with_suffix("")),
    "zip",
    root_dir=outputpath.parent,
    base_dir=outputpath.name,
)


with ZipFile(archivepath, "r") as archive:
    invalidfile = archive.testzip()

    if invalidfile is not None:
        raise RuntimeError(
            f"ZIP integrity check failed for: {invalidfile}"
        )

    includedfiles = archive.namelist()

archivesize = archivepath.stat().st_size / (1024 ** 2)

print("Model A export complete.")
print("Archive:", archivepath)
print(f"Archive size: {archivesize:.2f} MB")
print(f"Files included: {len(includedfiles)}")

print("\nArchive contents:")
for filename in includedfiles:
    print(filename)

Model A export complete.
Archive: /kaggle/working/modela_results_bundle.zip
Archive size: 5.45 MB
Files included: 27

Archive contents:
modela_outputs/
modela_outputs/checkpoints/
modela_outputs/figure4_consistency_histograms.png
modela_outputs/figure3_rotation_rank2_test9883.png
modela_outputs/figure2modela.png
modela_outputs/figure3_rotation_rank1_test9733.gif
modela_outputs/table4results.json
modela_outputs/figure3_mirror_rank1_test9269.png
modela_outputs/table3results.json
modela_outputs/figure3_mirror_rank2_test9347.png
modela_outputs/table1checks.json
modela_outputs/figure1_opticalflow.png
modela_outputs/figure3_selectedexamples.json
modela_outputs/table4perseed.json
modela_outputs/figure3_rotation_rank2_test9883.gif
modela_outputs/table3results.csv
modela_outputs/figure1_statistics.json
modela_outputs/table4results.csv
modela_outputs/figure3_mirror_rank1_test9269.gif
modela_outputs/table3perseed.json
modela_outputs/table2results.csv
modela_outputs/traininglog.json
modela_outputs